In [6]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None) 
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None) 

In [7]:
koi = pd.read_csv("datasets/KOI.csv", comment="#")
toi = pd.read_csv("datasets/TOI.csv", comment="#")
k2p = pd.read_csv("datasets/K2P.csv", comment="#")
print(f"KOI dataset shape: {koi.shape}")
print(f"TOI dataset shape: {toi.shape}")
print(f"K2P dataset shape: {k2p.shape}")

KOI dataset shape: (9564, 49)
TOI dataset shape: (7890, 65)
K2P dataset shape: (4024, 94)


In [18]:
koi_drop_subset = ['kepoi_name', 'kepler_name', 'koi_pdisposition', 'koi_score', 'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec', 'koi_time0bk', 'koi_time0bk_err1', 'koi_time0bk_err2', 'koi_tce_plnt_num', 'koi_tce_delivname', 'ra', 'dec', 'koi_teq_err1', 'koi_teq_err2']
koi2 = koi.drop(koi_drop_subset, axis=1)
koi2.columns

Index(['kepid', 'koi_disposition', 'koi_period', 'koi_period_err1',
       'koi_period_err2', 'koi_impact', 'koi_impact_err1', 'koi_impact_err2',
       'koi_duration', 'koi_duration_err1', 'koi_duration_err2', 'koi_depth',
       'koi_depth_err1', 'koi_depth_err2', 'koi_prad', 'koi_prad_err1',
       'koi_prad_err2', 'koi_teq', 'koi_insol', 'koi_insol_err1',
       'koi_insol_err2', 'koi_model_snr', 'koi_steff', 'koi_steff_err1',
       'koi_steff_err2', 'koi_slogg', 'koi_slogg_err1', 'koi_slogg_err2',
       'koi_srad', 'koi_srad_err1', 'koi_srad_err2', 'koi_kepmag'],
      dtype='object')

In [19]:
koi_full = pd.read_csv("datasets/KOI_full.csv", comment="#")
toi_full = pd.read_csv("datasets/TOI_full.csv", comment="#")
k2p_full = pd.read_csv("datasets/K2P_full.csv", comment="#")
print(f"KOI dataset shape: {koi_full.shape}")
print(f"TOI dataset shape: {toi_full.shape}")
print(f"K2P dataset shape: {k2p_full.shape}")

KOI dataset shape: (9564, 141)
TOI dataset shape: (7890, 87)
K2P dataset shape: (4024, 296)


In [21]:
k2p_full_default = k2p_full[k2p_full['default_flag'] == 1]
print(f"K2P dataset with default parameters shape: {k2p_full_default.shape}")

K2P dataset with default parameters shape: (1806, 296)


koi_full dataset and a k2p_full_default datasets will be used for analysis. koi_full dataset will be grouped by ```kepid``` and used for training, validating and testing, while k2p_full_default has already been filtered for duplicates and will only be used for testing. toi_full will not be used, as it contains less features than the other two datasets, making it difficult to cross-analyse them.  
19 features have been chosen for analysis based on their meritorical significance. Their exact meaning and description can be found in the README of the project. These features will be further inspected for missingness and correlation. All features contain error values - these will also be handled appropriately.  

In [22]:
koi_columns_set = [
    "kepid", "koi_disposition",

    # Orbital period
    "koi_period", "koi_period_err1", "koi_period_err2",

    # Eccentricity + argument of periastron
    "koi_eccen", "koi_eccen_err1", "koi_eccen_err2",
    "koi_longp", "koi_longp_err1", "koi_longp_err2",

    # Transit geometry / shape
    "koi_impact", "koi_impact_err1", "koi_impact_err2",
    "koi_duration", "koi_duration_err1", "koi_duration_err2",
    "koi_depth", "koi_depth_err1", "koi_depth_err2",
    "koi_ror", "koi_ror_err1", "koi_ror_err2",
    "koi_incl", "koi_incl_err1", "koi_incl_err2",
    "koi_dor", "koi_dor_err1", "koi_dor_err2",

    # Derived planet parameters
    "koi_prad", "koi_prad_err1", "koi_prad_err2",
    "koi_sma", "koi_sma_err1", "koi_sma_err2",
    "koi_teq", "koi_teq_err1", "koi_teq_err2",
    "koi_insol", "koi_insol_err1", "koi_insol_err2",

    # Stellar parameters
    "koi_steff", "koi_steff_err1", "koi_steff_err2",
    "koi_slogg", "koi_slogg_err1", "koi_slogg_err2",
    "koi_smet", "koi_smet_err1", "koi_smet_err2",
    "koi_srad", "koi_srad_err1", "koi_srad_err2",
    "koi_smass", "koi_smass_err1", "koi_smass_err2",
    "koi_sage", "koi_sage_err1", "koi_sage_err2",
]
k2p_columns_set = [
    "pl_name", "disposition",
    # Orbital period
    "pl_orbper", "pl_orbpererr1", "pl_orbpererr2",

    # Eccentricity + argument of periastron
    "pl_orbeccen", "pl_orbeccenerr1", "pl_orbeccenerr2",
    "pl_orblper", "pl_orblpererr1", "pl_orblpererr2",

    # Transit geometry / shape
    "pl_imppar", "pl_impparerr1", "pl_impparerr2",
    "pl_trandur", "pl_trandurerr1", "pl_trandurerr2",
    "pl_trandep", "pl_trandeperr1", "pl_trandeperr2",
    "pl_ratror", "pl_ratrorerr1", "pl_ratrorerr2",
    "pl_orbincl", "pl_orbinclerr1", "pl_orbinclerr2",
    "pl_ratdor", "pl_ratdorerr1", "pl_ratdorerr2",

    # Derived planet parameters
    "pl_rade", "pl_radeerr1", "pl_radeerr2",
    "pl_orbsmax", "pl_orbsmaxerr1", "pl_orbsmaxerr2",
    "pl_eqt", "pl_eqterr1", "pl_eqterr2",
    "pl_insol", "pl_insolerr1", "pl_insolerr2",

    # Stellar parameters
    "st_teff", "st_tefferr1", "st_tefferr2",
    "st_logg", "st_loggerr1", "st_loggerr2",
    "st_met", "st_meterr1", "st_meterr2",
    "st_rad", "st_raderr1", "st_raderr2",
    "st_mass", "st_masserr1", "st_masserr2",
    "st_age", "st_ageerr1", "st_ageerr2",
]